In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import parcels
import uxarray as ux
import xarray as xr
from uxarray.constants import INT_FILL_VALUE

In [ ]:
# no2d = xr.open_dataset('/work/bk1450/b383184/ELPHE_hackathon/Double_gyre/FESOM_v2.7/1d/unod.fesom.1950.nc')
# no2d

mesh = xr.open_dataset(
    "/work/bk1450/b383184/ELPHE_hackathon/Double_gyre/FESOM_v2.7/1d/fesom.mesh.diag.nc"
)
mesh

In [ ]:
grid_path = (
    "/work/bk1450/b383184/ELPHE_hackathon/Double_gyre/FESOM_v2.7/1d/fesom.mesh.diag.nc"
)

data_paths = [
    "/work/bk1450/b383184/ELPHE_hackathon/FESOM_1d/u.fesom.1950.nc",
    "/work/bk1450/b383184/ELPHE_hackathon/FESOM_1d/v.fesom.1950.nc",
    "/work/bk1450/b383184/ELPHE_hackathon/FESOM_1d/w.fesom.1950.nc",
]

ds = ux.open_mfdataset(grid_path, data_paths).compute()
ds

In [ ]:
((ds.v.isel(time=0, nz1=0) ** 2 + ds.u.isel(time=0, nz1=0) ** 2) ** 0.5).plot()

In [ ]:
ds = ds.rename_vars({"u": "U", "v": "V", "w": "W"})
ds

In [ ]:
ds = parcels.convert.fesom_to_ugrid(ds)
print("dims:", dict(ds.sizes))

In [ ]:
fieldset = parcels.FieldSet.from_ugrid_conventions(ds, mesh="spherical")

for name, field in fieldset.fields.items():
    interp = getattr(field, "interp_method", None)
    interp_name = interp.__name__ if interp is not None else "-"
    print(f"{name:>4s}  ->  {type(field).__name__:<11s}  interp={interp_name}")

In [ ]:
# Cell-view linear least-squares reconstruction for face-registered, layer-centered data.
def LinearFaceRecon(particle_positions, grid_positions, field):
    ti = grid_positions["T"]["index"]
    zi = grid_positions["Z"]["index"]
    fi = grid_positions["FACE"]["index"]

    uxg = field.grid.uxgrid
    gx = uxg._ds["gradient_coeff_x"].values[fi]  # (M, K)
    gy = uxg._ds["gradient_coeff_y"].values[fi]
    nb = uxg._ds["recon_face_neighbors"].values[fi]  # (M, K)
    mask = uxg._ds["recon_neighbor_mask"].values[fi]

    uc = field.data.values[ti, zi, fi]  # (M,)
    un = field.data.values[ti[:, None], zi[:, None], nb]  # (M, K)
    du = (un - uc[:, None]) * mask
    ax = np.sum(gx * du, axis=1)
    ay = np.sum(gy * du, axis=1)

    flon = uxg.face_lon.values[fi]
    flat = uxg.face_lat.values[fi]
    dx, dy = local_offsets(
        field.grid._mesh,
        flon,
        flat,
        particle_positions["lon"],
        particle_positions["lat"],
    )
    return uc + ax * dx + ay * dy

In [ ]:
DEG2M = 1852.0 * 60.0  # metres per degree (matches Ux_Velocity unit conversion)


def local_offsets(mesh, lon_c, lat_c, lon, lat):
    # local planar offset (dx, dy) of (lon, lat) from centroid (lon_c, lat_c)
    if mesh == "spherical":
        dlon = ((np.asarray(lon) - lon_c + 180.0) % 360.0) - 180.0
        dx = dlon * np.cos(np.deg2rad(lat_c)) * DEG2M
        dy = (np.asarray(lat) - lat_c) * DEG2M
    else:  # flat mesh: coordinates are already a planar (x, y)
        dx = np.asarray(lon) - lon_c
        dy = np.asarray(lat) - lat_c
    return dx, dy


# Compute and attach cell-view least-squares gradient coefficients. Adds to
# uxgrid._ds: gradient_coeff_x/_y, recon_face_neighbors (fill -> self),
# recon_neighbor_mask (1.0 valid / 0.0 absent), all (n_face, n_max_face_faces).
def build_reconstruction_terms(uxgrid, mesh):
    nbr = np.asarray(uxgrid.face_face_connectivity.values)  # (F, K)
    flon = np.asarray(uxgrid.face_lon.values)
    flat = np.asarray(uxgrid.face_lat.values)

    valid = nbr != INT_FILL_VALUE
    nbr_safe = np.where(valid, nbr, 0)

    dx, dy = local_offsets(
        mesh, flon[:, None], flat[:, None], flon[nbr_safe], flat[nbr_safe]
    )
    dx = np.where(valid, dx, 0.0)  # absent -> no contribution
    dy = np.where(valid, dy, 0.0)

    X2 = np.sum(dx * dx, axis=1)
    Y2 = np.sum(dy * dy, axis=1)
    XY = np.sum(dx * dy, axis=1)
    d = X2 * Y2 - XY * XY

    # cells with < 2 independent neighbours give d == 0 -> flat (zero-gradient)
    # fallback, i.e. the original piecewise-constant value.
    ok = d != 0.0
    inv_d = np.where(ok, 1.0 / np.where(ok, d, 1.0), 0.0)

    gx = (dx * Y2[:, None] - dy * XY[:, None]) * inv_d[:, None]
    gy = (dy * X2[:, None] - dx * XY[:, None]) * inv_d[:, None]
    gx = np.where(valid, gx, 0.0)
    gy = np.where(valid, gy, 0.0)

    dims = ("n_face", "n_max_face_faces")
    uxgrid._ds["gradient_coeff_x"] = xr.DataArray(gx, dims=dims)
    uxgrid._ds["gradient_coeff_y"] = xr.DataArray(gy, dims=dims)
    uxgrid._ds["recon_face_neighbors"] = xr.DataArray(nbr_safe, dims=dims)
    uxgrid._ds["recon_neighbor_mask"] = xr.DataArray(
        valid.astype(np.float64), dims=dims
    )
    return uxgrid

In [ ]:
# attach the metric terms to the grid shared by the fields
uxgrid = fieldset.fields["U"].grid.uxgrid
build_reconstruction_terms(uxgrid, mesh="spherical")
print("default T_face interpolator:", fieldset.fields["U"].interp_method.__name__)
print(
    "stored on uxgrid._ds:",
    [v for v in uxgrid._ds.data_vars if v.startswith(("gradient", "recon"))],
)

In [ ]:
fieldset.fields["U"].interp_method = LinearFaceRecon
fieldset.fields["V"].interp_method = LinearFaceRecon

In [ ]:
lon_grid, lat_grid = np.meshgrid(
    np.linspace(8.0, 11.0, 10), np.linspace(32.0, 33.0, 10)
)

In [ ]:
lon_grid.shape
z = np.full(lon_grid.size, 10.0)
z

In [ ]:
def run_fesom(label):
    # fesom.fields["U"].interp_method = interp
    # fesom.fields["V"].interp_method = interp
    lon_grid, lat_grid = np.meshgrid(
        np.linspace(8.0, 11.0, 10), np.linspace(32.0, 33.0, 10)
    )
    pset = parcels.ParticleSet(
        fieldset,
        pclass=parcels.Particle,
        lon=lon_grid.ravel(),
        lat=lat_grid.ravel(),
        z=np.full(lon_grid.size, 50.0),
    )
    out = parcels.ParticleFile(
        f"recon-{label}.parquet",
        outputdt=np.timedelta64(1, "h"),
        mode="w",
    )
    pset.execute(
        [parcels.kernels.AdvectionRK4],
        runtime=np.timedelta64(2, "D"),
        dt=np.timedelta64(5, "m"),
        output_file=out,
        verbose_progress=True,
    )
    return parcels.read_particlefile(f"recon-{label}.parquet")


df_const = run_fesom("LinearFaceRecon")
# df_lin = run_fesom("linear")
print("advection done")

In [ ]:
df_const

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df_const["lon"].to_numpy(), df_const["lat"].to_numpy(), 1)
# df_const['lon'].to_numpy(